# Streaming i asynchroniczność – zadania

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

1. Zbuduj pipeline z notebooka 3 od zera: zdefiniuj `State`, napisz trzy node'y (`build_prompt_node`, `call_llm_node`, `format_output_node`), skompiluj graf.

---
(czas: 15 min.)

In [3]:
import time
from typing import TypedDict
from langchain_core.runnables import RunnableConfig
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    question: str
    prompt_messages: list[dict]
    llm_response: str
    formatted_output: str


def build_prompt_node(state: State, config: RunnableConfig):
    time.sleep(3)
    configurable = config.get("configurable", {})
    system_prompt = configurable.get(
        "system_prompt",
        "Jesteś pomocnym asystentem. Odpowiadaj zwięźle i rzeczowo po polsku."
    )
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": state["question"]},
    ]
    return {"prompt_messages": messages}


def call_llm_node(state: State, config: RunnableConfig):
    time.sleep(4)
    configurable = config.get("configurable", {})
    model_name = configurable.get("model_name", "openai/gpt-4o-mini")
    temperature = configurable.get("temperature", 0.7)

    llm = ChatOpenAI(model=model_name, temperature=temperature)
    response = llm.invoke(state["prompt_messages"])
    return {"llm_response": response.content}


def format_output_node(state: State, config: RunnableConfig):
    time.sleep(2)
    configurable = config.get("configurable", {})
    model_name = configurable.get("model_name", "openai/gpt-4o-mini")
    output = (
        f"=== Odpowiedź modelu ({model_name}) ===\n"
        f"Pytanie: {state['question']}\n"
        f"Odpowiedź:\n{state['llm_response']}"
    )
    return {"formatted_output": output}


graph = StateGraph(State)

graph.add_node("build_prompt", build_prompt_node)
graph.add_node("call_llm", call_llm_node)
graph.add_node("format_output", format_output_node)

graph.add_edge(START, "build_prompt")
graph.add_edge("build_prompt", "call_llm")
graph.add_edge("call_llm", "format_output")
graph.add_edge("format_output", END)

pipeline = graph.compile()

initial_state = {"question": "Czym jest LangGraph i do czego się go używa?"}

runtime_config = {
    "configurable": {
        "model_name": "openai/gpt-4o-mini",
        "temperature": 0.5,
        "system_prompt": "Jesteś ekspertem od frameworków AI. Odpowiadaj zwięźle po polsku.",
    }
}

2. Wywołaj pipeline metodą `invoke`. Zmierz czas wykonania i wypisz cały wynikowy State.

---
(czas: 3 min.)

In [4]:
print("invoke — start")
start = time.time()

result = pipeline.invoke(initial_state, config=runtime_config)

print(f"invoke — wynik po {time.time() - start:.1f}s\n")
for key, value in result.items():
    print(key, "\n", value, "\n====\n")

invoke — start
invoke — wynik po 12.0s

question 
 Czym jest LangGraph i do czego się go używa? 
====

prompt_messages 
 [{'role': 'system', 'content': 'Jesteś ekspertem od frameworków AI. Odpowiadaj zwięźle po polsku.'}, {'role': 'user', 'content': 'Czym jest LangGraph i do czego się go używa?'}] 
====

llm_response 
 LangGraph to framework do tworzenia aplikacji opartych na języku naturalnym, który umożliwia integrację różnych modeli językowych i narzędzi AI. Używa się go do budowy interaktywnych aplikacji, które mogą przetwarzać i generować tekst, analizować dane oraz wspierać różne zadania związane z NLP (przetwarzanie języka naturalnego). LangGraph ułatwia tworzenie złożonych przepływów pracy i interakcji z użytkownikami. 
====

formatted_output 
 === Odpowiedź modelu (openai/gpt-4o-mini) ===
Pytanie: Czym jest LangGraph i do czego się go używa?
Odpowiedź:
LangGraph to framework do tworzenia aplikacji opartych na języku naturalnym, który umożliwia integrację różnych modeli językow

3. Wywołaj pipeline metodą `ainvoke`. Zmierz czas wykonania i wypisz cały wynikowy State.

---
(czas: 3 min.)

In [5]:
print("ainvoke — start")
start = time.time()

result = await pipeline.ainvoke(initial_state, config=runtime_config)

print(f"ainvoke — wynik po {time.time() - start:.1f}s\n")
for key, value in result.items():
    print(key, "\n", value, "\n====\n")

ainvoke — start
ainvoke — wynik po 11.4s

question 
 Czym jest LangGraph i do czego się go używa? 
====

prompt_messages 
 [{'role': 'system', 'content': 'Jesteś ekspertem od frameworków AI. Odpowiadaj zwięźle po polsku.'}, {'role': 'user', 'content': 'Czym jest LangGraph i do czego się go używa?'}] 
====

llm_response 
 LangGraph to framework do tworzenia aplikacji opartych na języku naturalnym, który umożliwia łatwe łączenie różnych modeli AI i komponentów przetwarzania języka. Używa się go do budowy rozwiązań takich jak chatboty, systemy rekomendacji czy aplikacje do analizy tekstu, umożliwiając integrację z różnymi źródłami danych i modelami językowymi. 
====

formatted_output 
 === Odpowiedź modelu (openai/gpt-4o-mini) ===
Pytanie: Czym jest LangGraph i do czego się go używa?
Odpowiedź:
LangGraph to framework do tworzenia aplikacji opartych na języku naturalnym, który umożliwia łatwe łączenie różnych modeli AI i komponentów przetwarzania języka. Używa się go do budowy rozwiązań ta

4. Wywołaj pipeline metodą `stream`. Dla każdego chunku wypisz nazwę node'a, czas od startu i zwrócone wartości.

---
(czas: 4 min.)

In [6]:
print("stream — start\n")
start = time.time()

for chunk in pipeline.stream(initial_state, config=runtime_config):
    node_name = list(chunk.keys())[0]
    elapsed = time.time() - start
    print(f"[{elapsed:.1f}s] Node '{node_name}' zakończony — output:")
    for key, value in chunk[node_name].items():
        print(f"  {key}: {value}")
    print()

stream — start

[3.0s] Node 'build_prompt' zakończony — output:
  prompt_messages: [{'role': 'system', 'content': 'Jesteś ekspertem od frameworków AI. Odpowiadaj zwięźle po polsku.'}, {'role': 'user', 'content': 'Czym jest LangGraph i do czego się go używa?'}]

[9.7s] Node 'call_llm' zakończony — output:
  llm_response: LangGraph to framework, który umożliwia tworzenie aplikacji opartych na języku naturalnym, integrując różne modele AI i źródła danych. Używa się go do budowy interaktywnych systemów, które potrafią przetwarzać, analizować i generować tekst w sposób bardziej złożony i kontekstowy. Dzięki LangGraph można łatwo łączyć różne komponenty AI, co pozwala na rozwijanie aplikacji w obszarze przetwarzania języka naturalnego.

[11.7s] Node 'format_output' zakończony — output:
  formatted_output: === Odpowiedź modelu (openai/gpt-4o-mini) ===
Pytanie: Czym jest LangGraph i do czego się go używa?
Odpowiedź:
LangGraph to framework, który umożliwia tworzenie aplikacji opartych na języku

5. Wywołaj pipeline metodą `astream`. Dla każdego chunku wypisz nazwę node'a, czas od startu i zwrócone wartości.

---
(czas: 4 min.)

In [7]:
print("astream — start\n")
start = time.time()

async for chunk in pipeline.astream(initial_state, config=runtime_config):
    node_name = list(chunk.keys())[0]
    elapsed = time.time() - start
    print(f"[{elapsed:.1f}s] Node '{node_name}' zakończony — output:")
    for key, value in chunk[node_name].items():
        print(f"  {key}: {value}")
    print()

astream — start

[3.0s] Node 'build_prompt' zakończony — output:
  prompt_messages: [{'role': 'system', 'content': 'Jesteś ekspertem od frameworków AI. Odpowiadaj zwięźle po polsku.'}, {'role': 'user', 'content': 'Czym jest LangGraph i do czego się go używa?'}]

[8.7s] Node 'call_llm' zakończony — output:
  llm_response: LangGraph to framework do budowy aplikacji opartych na przetwarzaniu języka naturalnego (NLP) oraz grafach wiedzy. Umożliwia tworzenie rozwiązań, które integrują różne źródła danych i modele AI, ułatwiając analizę i generowanie informacji. Jest używany w zadaniach takich jak eksploracja danych, rekomendacje oraz interaktywne systemy dialogowe.

[10.7s] Node 'format_output' zakończony — output:
  formatted_output: === Odpowiedź modelu (openai/gpt-4o-mini) ===
Pytanie: Czym jest LangGraph i do czego się go używa?
Odpowiedź:
LangGraph to framework do budowy aplikacji opartych na przetwarzaniu języka naturalnego (NLP) oraz grafach wiedzy. Umożliwia tworzenie rozwiązań, któ